In [0]:
financial corrupt dataset:

transaction_id	customer_id	txn_date	amount	currency	account_no	status	merchant
TX001	C101	2026-01-05	250.50	USD	1234567890	SUCCESS	Amazon
TX002	NULL	2026/01/06	-500	USD	12345	SUCCESS	Walmart
TX003	C103	invalid	abc	EUR	9876543210	DONE	Apple
TX001	C101	2026-01-05	250.50	USD	1234567890	SUCCESS	Amazon
NULL	C104	2026-01-08	100000000	INR	2222222222	FAILED	NULL

Common Finance Data Quality Errors to Remove
1. Null Values
Missing transaction_id
Missing customer_id
2. Duplicate Transactions
Same transaction_id repeated
3. Invalid Date Format
2026/01/06
invalid
4. Incorrect Data Types
amount = "abc"
5. Negative Transaction Amount
amount < 0
6. Outlier Detection
unusually large amount
7. Invalid Currency Codes
only allow:
USD
EUR
GBP
INR
8. Account Number Validation
length = 10 digits
9. Status Standardization
Convert:
DONE → SUCCESS
completed → SUCCESS
10. Text Standardization
trim spaces
uppercase

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Finance_Data_Cleaning") \
    .getOrCreate()

# -------------------------
# Sample Raw Data
# -------------------------

data = [
("TX001","C101","2026-01-05","250.50","USD","1234567890","SUCCESS","Amazon"),
("TX002",None,"2026/01/06","-500","USD","12345","SUCCESS","Walmart"),
("TX003","C103","invalid","abc","EUR","9876543210","DONE","Apple"),
("TX001","C101","2026-01-05","250.50","USD","1234567890","SUCCESS","Amazon"),
(None,"C104","2026-01-08","100000000","INR","2222222222","FAILED",None)
]

columns = [
"transaction_id",
"customer_id",
"txn_date",
"amount",
"currency",
"account_no",
"status",
"merchant"
]

df = spark.createDataFrame(data, columns)

# -------------------------
# STEP 1 — Remove whitespace
# -------------------------

for c in df.columns:
    df = df.withColumn(
        c,
        trim(col(c))
    )

# -------------------------
# STEP 2 — Standardize Text
# -------------------------

df = (
    df
    .withColumn("currency", upper("currency"))
    .withColumn("status", upper("status"))
)

# -------------------------
# STEP 3 — Fix Status Values
# -------------------------

df = (
    df
    .withColumn(
        "status",
        when(col("status")=="DONE","SUCCESS")
        .otherwise(col("status"))
    )
)

# -------------------------
# STEP 4 — Parse Date
# -------------------------

df = (
    df
    .withColumn(
        "txn_date",
        coalesce(
            to_date("txn_date","yyyy-MM-dd"),
            to_date("txn_date","yyyy/MM/dd")
        )
    )
)

# -------------------------
# STEP 5 — Convert Amount
# -------------------------

df = (
    df
    .withColumn(
        "amount",
        col("amount").cast("double")
    )
)

# -------------------------
# STEP 6 — Remove Null Keys
# -------------------------

df = (
    df
    .filter(
        col("transaction_id").isNotNull()
    )
    .filter(
        col("customer_id").isNotNull()
    )
)

# -------------------------
# STEP 7 — Remove Duplicates
# -------------------------

df = df.dropDuplicates(
    ["transaction_id"]
)

# -------------------------
# STEP 8 — Validate Amount
# -------------------------

df = (
    df
    .filter(col("amount").isNotNull())
    .filter(col("amount") >= 0)
    .filter(col("amount") <= 1000000)
)

# -------------------------
# STEP 9 — Currency Validation
# -------------------------

valid_currency = [
"USD",
"EUR",
"GBP",
"INR"
]

df = (
    df
    .filter(
        col("currency")
        .isin(valid_currency)
    )
)

# -------------------------
# STEP 10 — Account Validation
# -------------------------

df = (
    df
    .filter(
        length("account_no")==10
    )
)

# -------------------------
# STEP 11 — Merchant Cleanup
# -------------------------

df = (
    df
    .fillna({
        "merchant":"UNKNOWN"
    })
)

# -------------------------
# STEP 12 — Data Quality Flags
# -------------------------

validated_df = (
    df
    .withColumn(
        "quality_status",
        lit("VALID")
    )
)

validated_df.show(truncate=False)